# EDA corregido: aptitud solar fotovoltaica (Entrega 3)

Proyecto final, Machine Learning, Pregrado en Ciencia de Datos.
Integrantes: Jesús David Arévalo Montilla, Enmanuel David Díaz Molinares, Alex David Terán Meza.

Dataset: *Global Dataset of Solar Power Plants* (Mantilla-Guerra et al., 2026, arXiv:2603.20601),
disponible en [`cimejia/solarPV`](https://github.com/cimejia/solarPV).

Este cuaderno retoma la Entrega 1 y corrige los hallazgos que marcamos como error crítico en la
revisión previa del proyecto, tal como pide la guía de la Entrega 3 en su sección 7.1. Antes de
construir los 140 modelos de clasificación y regresión que exige la guía, necesitamos que el target
esté bien definido, que los problemas de calidad de datos estén tratados y no solo contados, y que
el esquema de validación cruzada sea el correcto para este dataset.

El paper fuente define la aptitud solar con la ecuación 1 (página 11):

$$IAS = 0.40\cdot Slope + 0.25\cdot Aspect + 0.20\cdot Hillshade + 0.15\cdot Curvature$$

Las cuatro capas de entrada se clasificaron según FAO (1976) para la pendiente, Wilson y Gallant
(2000) para la curvatura, y se combinaron con ponderación tipo AHP (Saaty, 1980), el mismo método
jerárquico de criterios que usan los propios autores del dataset. Dicho de otro modo: el índice es
topográfico de principio a fin. No entra irradiancia, ni temperatura, ni viento, ni humedad. Vamos a
usarlo de todas formas, porque el planteamiento del curso pide un target de clasificación y uno de
regresión, y aquí tenemos ambos en la misma variable: continua para regresión (`solar_aptitude`) y
discretizada en Baja, Media y Alta para clasificación (`solar_aptittude_class`). Lo que no podemos
hacer es llamarlo "aptitud solar" sin aclarar que mide terreno, no sol. Esa aclaración se desarrolla
con evidencia en la sección 9.

## Pipeline

De los datos crudos a la base limpia que usan los modelos.

![Pipeline](fig_pipeline_eda_mundial.png)

## 1. Librerías y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, StratifiedGroupKFold, cross_val_score
from sklearn.metrics import r2_score

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (8, 4.5)

CLASS_ORDER = ["Baja", "Media", "Alta"]
CLASS_COLORS = {"Baja": "#e76f51", "Media": "#e9c46a", "Alta": "#2a9d8f"}
REGION_ORDER = ["Américas", "Europa/África/M.Oriente", "Asia/Oceanía"]

## 2. Carga de datos

La Entrega 1 documentó la carga con `pandas.read_excel` sobre un archivo `.xlsx`. El archivo real
del proyecto es un CSV con separador `;`, decimal `,` y BOM UTF-8. Con los parámetros por defecto de
pandas, esas tres cosas juntas producen una sola columna de texto en vez de 29 columnas numéricas, así
que el ETL original no era reproducible con los datos que en realidad usamos.

In [ ]:
DATA_PATH = "../Dataset_Mundial_Final(2).csv"

df_raw = pd.read_csv(DATA_PATH, sep=";", decimal=",", encoding="utf-8-sig")
print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")
df_raw.head(3)

In [ ]:
schema = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_nulos": df_raw.isna().sum().values,
    "pct_nulos": (df_raw.isna().mean().values * 100).round(2),
})
schema

Una sola columna tiene nulos de verdad: `area`, con 27.7 % de faltantes. El resto del esquema carga
limpio. Esto ya contradice el diagnóstico de la Entrega 1, que daba por buena toda la tabla sin
reportar este hueco. La razón del faltante se explica en el punto D6 más abajo, junto con por qué no
conviene imputarlo a la ligera.

In [ ]:
na_pct = df_raw.isna().mean().sort_values(ascending=False)
na_pct = na_pct[na_pct > 0]
fig, ax = plt.subplots(figsize=(6, 2))
na_pct.plot(kind="barh", ax=ax, color="#e76f51")
ax.set_xlabel("Proporción de nulos")
ax.set_title("Única columna con faltantes en el CSV crudo")
plt.tight_layout()
plt.show()

## 3. Variable objetivo

`solar_aptitude` va de 0 a 0.942, con media 0.68. Al discretizarla en los umbrales 0.4 y 0.6 definidos
por el paper (sección 2.4.6), la clase Alta queda con el 75 % de las plantas, Media con el 22 % y Baja
con apenas el 3 %. Ese desbalance no es un defecto del muestreo: viene de que la mayoría de las plantas
del dataset ya están construidas en terrenos que el propio índice califica como favorables, algo
esperable si quien construye una planta solar primero revisa la pendiente del terreno. El problema
real no es que Baja sea minoritaria, sino de dónde sale esa minoría, que se investiga en la sección 9.

In [ ]:
print(df_raw["solar_aptitude"].describe())
print()
dist = df_raw["solar_aptittude_class"].value_counts().reindex(CLASS_ORDER)
dist_pct = (dist / len(df_raw) * 100).round(2)
pd.DataFrame({"n": dist, "%": dist_pct})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(df_raw["solar_aptitude"], bins=50, ax=axes[0], color="#2a9d8f")
axes[0].axvline(0.4, color="crimson", ls="--", lw=1)
axes[0].axvline(0.6, color="crimson", ls="--", lw=1)
axes[0].set_title("solar_aptitude (target de regresión)")
axes[0].set_xlabel("IAS")

sns.barplot(x=dist.index, y=dist.values, ax=axes[1], hue=dist.index,
            palette=CLASS_COLORS, legend=False)
for i, (n, p) in enumerate(zip(dist.values, dist_pct.values)):
    axes[1].text(i, n, f"{n:,}\n({p}%)", ha="center", va="bottom", fontsize=9)
axes[1].set_title("solar_aptittude_class (target de clasificación)")

plt.tight_layout()
plt.savefig("fig_distribucion_target_e3.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Calidad de datos

La Entrega 1 reportó "una única violación" en la validación de rangos. Eso era incorrecto: hay al
menos doce problemas distintos, cada uno con una causa identificable en la metodología del paper
(secciones 2.4.3 a 2.4.6). Los numeramos D1 a D14 para poder referenciarlos luego, cuando se decide
qué hacer con cada predictora.

### D1. Filas sin cobertura de DEM

El modelo digital de elevación (DEM) que usa el paper viene organizado por continente y con
resolución de 9 arco-segundos. Fuera de esa cobertura, `slope`, `aspect`, `curvature` y `elevation`
quedan en cero, y como esas cuatro variables alimentan directamente la ecuación 1, el IAS también
sale en cero, lo que la clasifica como Baja sin que eso diga nada real sobre el terreno.

In [ ]:
mask_d1 = (
    (df_raw["solar_aptitude"] == 0)
    & (df_raw["slope"] == 0)
    & (df_raw["aspect"] == 0)
    & (df_raw["curvature"] == 0)
    & (df_raw["elevation"] == 0)
)
d1 = df_raw[mask_d1]
n_baja_total = (df_raw["solar_aptittude_class"] == "Baja").sum()
print(f"Filas sin cobertura de DEM: {len(d1)}")
print(f"Todas con clase 'Baja': {(d1['solar_aptittude_class'] == 'Baja').all()}")
print(f"Proporción de la clase Baja total que representan: {len(d1) / n_baja_total * 100:.1f}%")
d1["country"].value_counts().head(8)

Son casi todas islas pequeñas (Reunión, Mauricio, Cabo Verde) donde el DEM regional no tiene
resolución suficiente. No son plantas con mal terreno: son plantas sin dato de terreno. La decisión
es eliminarlas, no imputarlas.

### D2. Filas sin cobertura de raster GSA

Lo mismo pasa con el raster del Global Solar Atlas que alimenta `ghi` y `pv_potential`. Cuando no hay
cobertura, ambas quedan en cero en vez de en nulo, lo que las hace indistinguibles de un valor real
bajo si no se revisan a mano.

In [ ]:
mask_d2 = (df_raw["ghi"] == 0) & (df_raw["pv_potential"] == 0)
mask_wind0 = df_raw["wind_speed"] == 0
d2 = df_raw[mask_d2]
print(f"Filas con ghi=pv_potential=0: {len(d2)}")
print(d2[["country", "ghi", "humidity", "pv_potential", "wind_speed", "solar_aptitude"]])
print(f"\nFilas adicionales con wind_speed=0: {mask_wind0.sum()}")
df_raw.loc[mask_wind0, ["country", "wind_speed", "solar_aptitude"]]

### D3. `aspect = -1`

Esto no es un error, es la convención del paper para marcar terreno plano, donde la orientación no
está definida. Lo confirmamos cruzándolo con la columna categórica y con la pendiente.

In [ ]:
mask_d3 = df_raw["aspect"] == -1
d3 = df_raw[mask_d3]
print(f"aspect = -1: {len(d3)} filas ({len(d3) / len(df_raw) * 100:.2f}%)")
print(f"  Coincide con aspect_type == 'Flat': {(d3['aspect_type'] == 'Flat').mean() * 100:.1f}%")
print(f"  Coincide con slope == 0: {(d3['slope'] == 0).mean() * 100:.1f}%")

Coincide al 100 % con ambas condiciones, así que el centinela es consistente y hay que tratarlo como
una categoría propia, no como un ángulo de 359 grados cualquiera. La sección 5 construye un indicador
binario `is_flat` justo para eso.

### D4. `dist_to_road` con valores que no caben en la metodología

El paper calcula esta distancia con la red vial de Sudamérica (GEOFABRIK) proyectada en "South
America Albers Equal Area", y admite en el texto que fuera de esa cobertura el cálculo se hizo de
forma genérica. Su propio máximo reportado es 6.31 km. En el CSV hay filas muy por encima de eso.

In [ ]:
print(df_raw["dist_to_road"].describe())
print()
for km in (6.31, 20, 100):
    n = (df_raw["dist_to_road"] > km * 1000).sum()
    print(f"  filas con dist_to_road > {km} km: {n}")

mask_d4 = df_raw["dist_to_road"] > 100_000
print("\nPaíses más frecuentes entre las filas con dist_to_road > 100 km:")
df_raw.loc[mask_d4, "country"].value_counts().head(8)

El máximo real es 2,296 km, en Pakistán, un país fuera de Sudamérica. Eso coincide con lo que el paper
mismo advierte: fuera de la región para la que se calibró la red vial, la distancia pierde sentido
físico. Tratamos más de 100 km como error de cálculo y lo convertimos en nulo en la sección 5, en vez
de dejarlo entrar al modelo como si fuera una distancia real.

### D5. `area` incoherente con `capacity`

Una planta solar típica instala entre 40 y 100 vatios por metro cuadrado, con un techo físico
alrededor de 200. Si dividimos `capacity` entre `area`, buena parte del dataset queda muy por encima
de eso.

In [ ]:
# capacity está en MW según GEM. El paper etiqueta la columna como kWp en su Tabla 2, pero esa
# unidad no es compatible con una mediana de varios MW en plantas de miles de m^2 (ver D13).
density_w_m2 = df_raw["capacity"] * 1_000_000 / df_raw["area"]
print("Densidad capacidad/área, asumiendo capacity en MW (físico esperado: 40-200 W/m²):")
print(density_w_m2.describe())
print(f"\nPlantas con densidad > 300 W/m²: {(density_w_m2 > 300).sum()}")
print(f"area máxima registrada: {df_raw['area'].max():,.1f} m²")

El propio procedimiento del paper explica por qué: `area` sale de unir cada coordenada con el
polígono de OpenStreetMap más cercano etiquetado como instalación solar. Si el punto cae sobre un
solo panel o un techo en vez de sobre la planta completa, el área queda diminuta y la densidad se
dispara. No es ruido aleatorio, es un artefacto sistemático del método de unión espacial, así que
`area` no sirve como medida confiable del tamaño físico de una planta.

### D6 y D7. `area` faltante y lo que ese faltante revela

In [ ]:
na_area = df_raw["area"].isna()
print(f"NaN en area: {na_area.sum()} ({na_area.mean() * 100:.2f}%)")
print("\nsize para filas con area faltante:")
print(df_raw.loc[na_area, "size"].value_counts())

print("\nComposición de clase según disponibilidad de area:")
rate = (
    df_raw.assign(area_na=na_area)
    .groupby("area_na")["solar_aptittude_class"]
    .value_counts(normalize=True)
    .unstack()[CLASS_ORDER]
    .round(3) * 100
)
rate

Cuando `area` falta, el 90 % de esas plantas son clase Alta, frente al 70 % cuando sí hay dato. El
faltante no es aleatorio: está correlacionado con la clase. También vemos que la columna `size`
asigna "Small" a las 16,329 filas sin área, mezclando plantas que de verdad son pequeñas con plantas
de las que simplemente no se pudo calcular el área. Eso hace que `size`, tal como viene en el dataset,
no sea una variable confiable por sí sola.

### D8. Duplicados

La Entrega 1 afirmó que no había duplicados. Revisando atributos completos, sin contar el
identificador, el nombre o el código, la afirmación no se sostiene.

In [ ]:
feature_cols = [c for c in df_raw.columns if c not in ("OBJECTID", "code", "plant_name")]

exact_dup = df_raw.duplicated().sum()
attr_dup = df_raw.duplicated(subset=feature_cols, keep=False).sum()
print(f"Duplicados exactos (fila completa, incluye ID): {exact_dup}")
print(f"Duplicados de atributos (sin ID/código/nombre): {attr_dup}")

coord_dup_mask = df_raw.duplicated(subset=["latitude", "longitude"], keep=False)
n_coord_groups = df_raw.loc[coord_dup_mask].groupby(["latitude", "longitude"]).ngroups
print(f"\nFilas con coordenadas repetidas: {coord_dup_mask.sum()} en {n_coord_groups} grupos")

class_per_coord = (
    df_raw.loc[coord_dup_mask]
    .groupby(["latitude", "longitude"])["solar_aptittude_class"]
    .nunique()
)
print(f"Grupos de coordenada con más de una clase distinta: {(class_per_coord > 1).sum()}")

Ningún par de filas es idéntico de principio a fin, por eso el chequeo ingenuo de duplicados exactos
da cero y parece confirmar lo que decía la Entrega 1. Pero al excluir solo las columnas de
identificación, aparecen más de 1,500 filas que comparten todo lo demás, y casi 10,000 filas comparten
coordenada con al menos otra planta. En ocho de esos grupos, la misma coordenada tiene dos clases
distintas, lo que confirma que parte del ruido no viene del terreno sino de cómo se construyó el
dataset.

### D9 y D10. Ruido en el propio target

In [ ]:
for edge in (0.4, 0.6):
    near = df_raw[np.isclose(df_raw["solar_aptitude"], edge, atol=0.0005)]
    print(f"solar_aptitude ≈ {edge}: {len(near)} filas -> {near['solar_aptittude_class'].value_counts().to_dict()}")

expected_rounded = (df_raw["solar_aptitude"] * 10).round().astype(int)
mismatch = (expected_rounded != df_raw["solar_aptitude_rounded"]).sum()
print(f"\nInconsistencias de redondeo en solar_aptitude_rounded: {mismatch} de {len(df_raw):,} filas")

Justo en los umbrales de corte hay filas con el mismo valor de IAS clasificadas en clases distintas,
porque el binning se hizo antes de redondear a tres decimales. Esto es ruido de etiqueta irreducible:
no se puede corregir con limpieza, solo documentar como techo del desempeño que cualquier modelo puede
alcanzar cerca de 0.4 y 0.6. `solar_aptitude_rounded` tiene además 255 inconsistencias de redondeo
propias, así que queda fuera de las predictoras y de los targets.

### D11. `wind_speed` sesgada a la baja

In [ ]:
df_raw["wind_speed"].describe()

El paper calcula esta variable a partir de los componentes zonal y meridional del viento (`u10`,
`v10`), ya promediados mensualmente, antes de obtener la rapidez. Promediar direcciones opuestas
dentro del mismo mes cancela parte de la magnitud real del viento. El resultado no es la velocidad
media del viento en el sitio, es la magnitud del viento vectorial medio anual, que por construcción
es menor. Hay que leerla con ese matiz, no como un sensor de viento real.

### D12. Redundancia categórica

`slope_type`, `curvature_type`, `aspect_type` y `dt_wind` son clasificaciones derivadas de su
variable numérica correspondiente, con rangos que no se solapan.

In [ ]:
print(df_raw.groupby("slope_type")["slope"].agg(["min", "max", "count"]).sort_values("min"))
print()
print(df_raw.groupby("curvature_type")["curvature"].agg(["min", "max", "count"]).sort_values("min"))

Incluir la versión numérica y la categórica como predictoras separadas no añade información nueva,
solo infla la dimensionalidad y, en modelos lineales, el VIF. Usamos la numérica y descartamos su
versión categórica.

### D13. Los máximos del paper no coinciden con los del CSV

In [ ]:
paper_max = {"area": 153_488, "dist_to_road": 6_309.67, "capacity": 57.5}
for col, pmax in paper_max.items():
    within = (df_raw[col] <= pmax).mean() * 100
    print(f"{col:15s} máx. CSV = {df_raw[col].max():>14,.1f} | máx. paper = {pmax:>10,.1f} | "
          f"% de filas del CSV por debajo del máx. del paper = {within:.1f}%")

En las tres variables, el CSV tiene filas muy por encima del máximo que reporta el paper en su propia
sección 3. La explicación más simple es que las figuras del artículo están filtradas o recortadas
antes de graficarse, mientras que el CSV público no pasó por ese mismo filtro. Al citar los valores
del paper hay que aclarar esa diferencia.

### D14. Clima de baja resolución en algunos países

In [ ]:
for c in ["Argentina", "Colombia", "Chile"]:
    sub = df_raw[df_raw["country"] == c]
    print(f"{c:10s} n={len(sub):4d}  humidity únicos={sub['humidity'].nunique():3d}  "
          f"wind_direction únicos={sub['wind_direction'].nunique():3d}")

Argentina tiene 64 plantas pero solo 4 valores distintos de humedad. El clima viene de promedios
mensuales de ERA5 con resolución espacial gruesa, así que dentro de estos países el "clima" es casi
una constante compartida por decenas de plantas, no una medición local. Conviene tenerlo presente al
interpretar la importancia que SHAP le dé a estas variables en el cuaderno de modelos.

## 5. Construcción del dataset limpio

Eliminamos las filas centinela de D1 y D2, y deduplicamos atributos de D8. El resto de los problemas
(D3, D4, D5 a D7, D12) se resuelven con variables derivadas, no borrando filas, porque en esos casos
la fila sigue teniendo información real en sus demás columnas.

In [ ]:
drop_mask = mask_d1 | mask_d2
df_clean = df_raw.loc[~drop_mask].copy()
df_clean = df_clean.drop_duplicates(subset=feature_cols, keep="first")

print(f"Filas: {len(df_raw):,} -> {len(df_clean):,} tras D1+D2+D8 "
      f"({len(df_raw) - len(df_clean)} eliminadas)")
print()
print(df_clean["solar_aptittude_class"].value_counts())
print((df_clean["solar_aptittude_class"].value_counts(normalize=True) * 100).round(2))

In [ ]:
before = df_raw["solar_aptittude_class"].value_counts(normalize=True).reindex(CLASS_ORDER) * 100
after = df_clean["solar_aptittude_class"].value_counts(normalize=True).reindex(CLASS_ORDER) * 100
comp = pd.DataFrame({"Antes de limpiar": before, "Después de limpiar": after})

fig, ax = plt.subplots(figsize=(7, 4))
comp.plot(kind="bar", ax=ax, color=["#adb5bd", "#2a9d8f"])
ax.set_ylabel("% de filas")
ax.set_title("Proporción de clases antes y después de D1, D2 y D8")
plt.tight_layout()
plt.show()

El cambio es pequeño en términos relativos: la limpieza quita 1,002 filas de 58,978, y Baja pasa de
3.21 % a 3.15 %. No es una limpieza que resuelva el desbalance, pero sí evita que un 3.4 % de la clase
minoritaria sea, literalmente, ausencia de datos disfrazada de terreno malo.

In [ ]:
# D3: variables circulares para aspect, con indicador de plano
aspect_valid = df_clean["aspect"].where(df_clean["aspect"] != -1)
aspect_rad = np.deg2rad(aspect_valid)
df_clean["aspect_sin"] = np.sin(aspect_rad).fillna(0.0)
df_clean["aspect_cos"] = np.cos(aspect_rad).fillna(0.0)
df_clean["is_flat"] = (df_clean["aspect"] == -1).astype(int)

# wind_direction es circular por naturaleza, sin centinela que tratar aparte
wind_rad = np.deg2rad(df_clean["wind_direction"])
df_clean["wind_sin"] = np.sin(wind_rad)
df_clean["wind_cos"] = np.cos(wind_rad)

# D4: distancias > 100 km se tratan como error de proyección, no como dato real
df_clean["dist_to_road_clean"] = df_clean["dist_to_road"].where(df_clean["dist_to_road"] <= 100_000)
df_clean["log_dist_to_road"] = np.log1p(df_clean["dist_to_road_clean"])

# Macro-región por longitud, para las secciones 6, 9 y 10
def macro_region(lon):
    if lon < -30:
        return "Américas"
    if lon < 60:
        return "Europa/África/M.Oriente"
    return "Asia/Oceanía"

df_clean["macro_region"] = df_clean["longitude"].apply(macro_region)

print("Variables derivadas: aspect_sin, aspect_cos, is_flat, wind_sin, wind_cos, "
      "log_dist_to_road, macro_region")
df_clean[["aspect", "aspect_sin", "aspect_cos", "is_flat",
          "dist_to_road", "log_dist_to_road"]].describe()

Usar seno y coseno en lugar del ángulo en grados es la forma estándar de tratar variables circulares
en estadística: 1 y 359 grados son casi el mismo punto en la brújula, pero numéricamente están casi
tan separados como es posible (Fisher, 1993). Con el ángulo crudo, un modelo lineal o una prueba de
Kruskal-Wallis los trata como extremos opuestos, que es exactamente el error que corregimos aquí para
`aspect` y `wind_direction`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df_clean["dist_to_road"], bins=60, ax=axes[0], color="#e76f51")
axes[0].set_title("dist_to_road original")
sns.histplot(df_clean["log_dist_to_road"].dropna(), bins=60, ax=axes[1], color="#2a9d8f")
axes[1].set_title("log1p(dist_to_road), >100 km tratado como error")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(y=np.log1p(df_clean["area"].dropna()), ax=axes[0], color="#e76f51")
axes[0].set_title("log1p(area): outliers de D5")
axes[0].set_ylabel("log1p(area en m²)")
sns.boxplot(y=df_clean["log_dist_to_road"].dropna(), ax=axes[1], color="#e76f51")
axes[1].set_title("log1p(dist_to_road): outliers de D4")
axes[1].set_ylabel("log1p(dist_to_road en m)")
plt.tight_layout()
plt.show()

Incluso después de pasar a escala logarítmica, ambas variables conservan una cola larga de outliers.
Eso es coherente con lo encontrado en D4 y D5: no es una distribución naturalmente sesgada, es una
mezcla de mediciones buenas y mediciones con error de cálculo.

## 6. Análisis univariado corregido

### 6.1 Geográficas

Más que ver la latitud y la longitud por separado, interesa ver dónde cae cada clase en el mapa,
porque ahí es donde empieza a notarse el patrón que se investiga a fondo en la sección 9.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for cls in ["Alta", "Media", "Baja"]:
    sub = df_clean[df_clean["solar_aptittude_class"] == cls]
    ax.scatter(sub["longitude"], sub["latitude"], s=3, alpha=0.4,
               c=CLASS_COLORS[cls], label=cls)
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_title("Ubicación de cada planta, coloreada por clase de aptitud")
ax.legend(markerscale=4, title="Clase")
plt.tight_layout()
plt.savefig("fig_mapa_clases_e3.png", dpi=150, bbox_inches="tight")
plt.show()

La clase Baja (naranja) no se reparte de forma pareja por el mapa: se concentra casi toda en una
franja alrededor de Europa. Si el IAS midiera solo pendiente, orientación, sombra y curvatura, no
habría razón geográfica para eso. Volvemos sobre este punto con números en la sección 9.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(df_clean["latitude"], bins=50, ax=axes[0], color="#264653")
axes[0].set_title(f"Latitud (media {df_clean['latitude'].mean():.1f}°)")
sns.histplot(df_clean["longitude"], bins=50, ax=axes[1], color="#264653")
axes[1].set_title(f"Longitud (media {df_clean['longitude'].mean():.1f}°)")
df_clean["country"].value_counts().head(10).plot.barh(ax=axes[2], color="#264653")
axes[2].invert_yaxis()
axes[2].set_title("Países con más plantas")
plt.tight_layout()
plt.show()

### 6.2 Topográficas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.histplot(df_clean["elevation"], bins=50, ax=axes[0, 0], color="#2a9d8f")
axes[0, 0].set_title("Elevación (m)")
sns.histplot(df_clean["slope"], bins=50, ax=axes[0, 1], color="#2a9d8f")
axes[0, 1].set_title("Pendiente (°)")
sns.histplot(df_clean["curvature"], bins=50, ax=axes[1, 0], color="#2a9d8f")
axes[1, 0].set_title("Curvatura")
axes[1, 1].scatter(df_clean["aspect_cos"], df_clean["aspect_sin"], s=2, alpha=0.1, c="#2a9d8f")
axes[1, 1].set_title("Orientación (aspect) en seno y coseno")
axes[1, 1].set_xlabel("cos(aspect)")
axes[1, 1].set_ylabel("sin(aspect)")
plt.tight_layout()
plt.show()

Pendiente y curvatura están casi todas pegadas a cero: la inmensa mayoría de las plantas se construye
en terreno plano, algo lógico porque nivelar terreno inclinado cuesta dinero. El gráfico de
orientación en seno y coseno no tiene forma de anillo perfecto: hay más densidad hacia el sur, que es
la orientación que más sol recibe en el hemisferio norte, donde está la mayoría de las plantas.

### 6.3 Climáticas y logísticas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.histplot(df_clean["ambient_temperature"], bins=50, ax=axes[0, 0], color="#e76f51")
axes[0, 0].set_title("Temperatura ambiente (°C)")
sns.histplot(df_clean["humidity"], bins=50, ax=axes[0, 1], color="#e76f51")
axes[0, 1].set_title("Humedad (%)")
sns.histplot(df_clean["wind_speed"], bins=50, ax=axes[1, 0], color="#e76f51")
axes[1, 0].set_title("Viento vectorial medio (m/s), ver D11")
sns.histplot(df_clean["log_dist_to_road"].dropna(), bins=50, ax=axes[1, 1], color="#e76f51")
axes[1, 1].set_title("log1p(dist_to_road)")
plt.tight_layout()
plt.show()

## 7. Análisis bivariado corregido

La Entrega 1 usó correlación de Pearson sobre variables de cola pesada (`area`, `dist_to_road`) y
circulares (`aspect`, `wind_direction`) sin transformar. Pearson asume relaciones lineales y es
sensible a outliers, justo lo que sobra en esas columnas (Hollander, Wolfe y Chicken, 2014). Aquí
usamos Spearman, que trabaja sobre rangos y no asume linealidad, junto con las versiones seno y
coseno de las variables circulares.

In [ ]:
numeric_cols = [
    "latitude", "longitude", "elevation", "slope", "curvature",
    "aspect_sin", "aspect_cos", "log_dist_to_road", "ambient_temperature",
    "humidity", "wind_speed", "wind_sin", "wind_cos", "ghi", "optimal_tilt",
    "pv_potential", "solar_aptitude",
]
corr = df_clean[numeric_cols].corr(method="spearman")

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="RdBu_r", center=0, square=True, cbar_kws={"label": "Spearman r"})
plt.title("Matriz de correlación de Spearman")
plt.tight_layout()
plt.savefig("fig_correlacion_e3.png", dpi=150, bbox_inches="tight")
plt.show()

corr["solar_aptitude"].drop("solar_aptitude").sort_values(key=abs, ascending=False)

La correlación más fuerte con `solar_aptitude` no es con ninguna de las cuatro variables de la
ecuación 1. Es con `longitude`. Eso ya es una señal de que el índice no se comporta como su propia
fórmula promete, y es el hilo que seguimos en la sección 9.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for region, color in zip(REGION_ORDER, sns.color_palette("viridis", 3)):
    sub = df_clean[(df_clean["macro_region"] == region) & (df_clean["slope"] <= 20)]
    ax.scatter(sub["slope"], sub["solar_aptitude"], s=3, alpha=0.15, color=color, label=region)
ax.set_xlabel("Pendiente (°)")
ax.set_ylabel("solar_aptitude")
ax.set_title("Pendiente contra aptitud, por macro-región")
ax.legend(markerscale=6)
plt.tight_layout()
plt.show()

Dentro de cada región, los puntos sí muestran una tendencia negativa leve: más pendiente, menos
aptitud, que es lo que la física del problema predice. Pero las tres nubes de puntos están a alturas
distintas en el eje vertical. La relación local es real; lo que falta es una relación global que
conecte una región con otra. Eso es justo lo que un desplazamiento entre lotes de procesamiento
produciría.

## 8. Factor de inflación de varianza

El VIF mide cuánta varianza de una variable explica una combinación lineal del resto. Un valor por
encima de 5 suele tomarse como señal de colinealidad relevante, y por encima de 10 como señal fuerte,
aunque ese umbral de 10 es una convención sin base estadística firme y conviene tratarlo como guía, no
como regla dura (O'Brien, 2007).

In [ ]:
vif_cols = [
    "latitude", "longitude", "elevation", "slope", "curvature",
    "aspect_sin", "aspect_cos", "wind_sin", "wind_cos",
    "ambient_temperature", "humidity", "wind_speed", "ghi",
]
vif_data = df_clean[vif_cols].dropna()
X_vif = add_constant(vif_data)
vif_table = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
print(f"n = {len(vif_data):,} casos completos")
vif_table

Todas las predictoras quedan muy por debajo de 5, con `const` como la única excepción esperable
porque mide la varianza explicada por el intercepto, no por una variable real. Eso confirma que la
codificación seno y coseno no introdujo colinealidad extra, y que las 13 predictoras numéricas pueden
entrar juntas a un modelo lineal sin que la multicolinealidad distorsione sus coeficientes.

## 9. El IAS es topográfico y tiene un desplazamiento por región

Esto corrige tres afirmaciones de entregas anteriores a la vez: que el IAS mide terreno y clima
juntos, que la clase Baja se concentra en cuatro países europeos, y que su dominancia por longitud se
debe a sesgo de muestreo. Lo mostramos en tres pasos: primero, que el IAS no correlaciona con las
variables que su propia fórmula declara; segundo, que sí correlaciona con la posición geográfica;
tercero, que un modelo entrenado en un continente no predice bien en otro.

In [ ]:
print("Spearman(solar_aptitude, variables de la ecuación 1):")
for col in ["slope", "curvature"]:
    r, p = spearmanr(df_clean["solar_aptitude"], df_clean[col])
    print(f"  {col:10s} r={r:+.3f}  p={p:.2e}")

mask_valid = df_clean["aspect"] != -1
r_a, p_a = spearmanr(df_clean.loc[mask_valid, "solar_aptitude"], df_clean.loc[mask_valid, "aspect"])
print(f"  {'aspect':10s} r={r_a:+.3f}  p={p_a:.2e}  (excluyendo el centinela -1)")

r_lon, p_lon = spearmanr(df_clean["solar_aptitude"], df_clean["longitude"])
r_lat, p_lat = spearmanr(df_clean["solar_aptitude"], df_clean["latitude"])
print(f"\n  {'longitude':10s} r={r_lon:+.3f}  p={p_lon:.2e}")
print(f"  {'latitude':10s} r={r_lat:+.3f}  p={p_lat:.2e}")

La pendiente y la curvatura correlacionan muy poco con el IAS, y la orientación prácticamente nada.
La longitud, en cambio, correlaciona casi diez veces más fuerte que la pendiente. Una variable que no
entra en la fórmula del índice explica más de su variación que las que sí entran.

In [ ]:
region_summary = df_clean.groupby("macro_region").agg(n=("solar_aptitude", "size"),
                                                        media_IAS=("solar_aptitude", "mean"))
region_class = (df_clean.groupby("macro_region")["solar_aptittude_class"]
                .value_counts().unstack()[CLASS_ORDER])
region_summary.join(region_class).reindex(REGION_ORDER)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(data=df_clean, x="macro_region", y="solar_aptitude", order=REGION_ORDER,
            hue="macro_region", palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("solar_aptitude por macro-región")
axes[0].tick_params(axis="x", rotation=15)

props = (df_clean.groupby("macro_region")["solar_aptittude_class"]
         .value_counts(normalize=True).unstack()[CLASS_ORDER].reindex(REGION_ORDER))
props.plot(kind="bar", stacked=True, color=[CLASS_COLORS[c] for c in props.columns], ax=axes[1])
axes[1].set_ylabel("Proporción")
axes[1].set_title("Composición de clases por macro-región")
axes[1].legend(title="Clase", bbox_to_anchor=(1.02, 1), loc="upper left")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

Asia y Oceanía tienen una media de IAS casi 0.17 puntos por encima de las otras dos regiones, y casi
no tienen clase Baja. Europa, África y Medio Oriente concentran la gran mayoría de la clase Baja. Esto
ya corrige el error puntual de la Entrega 1, que atribuía Baja solo a cuatro países europeos: en
realidad son siete los que suman la mayoría, y el patrón es regional, no nacional.

In [ ]:
baja_by_country = (df_clean[df_clean["solar_aptittude_class"] == "Baja"]["country"]
                    .value_counts().head(10))
fig, ax = plt.subplots(figsize=(7, 4))
baja_by_country.plot(kind="barh", ax=ax, color="#e76f51")
ax.invert_yaxis()
ax.set_title("Países con más plantas en clase Baja")
plt.tight_layout()
plt.show()

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)


def cv_r2(features):
    sub = df_clean.dropna(subset=features + ["solar_aptitude"])
    X, y = sub[features].to_numpy(), sub["solar_aptitude"].to_numpy()
    scores = cross_val_score(HistGradientBoostingRegressor(random_state=SEED), X, y,
                              cv=kf, scoring="r2", n_jobs=-1)
    return scores.mean(), scores.std()

topo_only = ["slope", "aspect_sin", "aspect_cos", "curvature"]
topo_latlon = topo_only + ["latitude", "longitude"]

m1, s1 = cv_r2(topo_only)
m2, s2 = cv_r2(topo_latlon)
print(f"R² CV, HistGradientBoosting, solo topografía:    {m1:.3f} ± {s1:.3f}")
print(f"R² CV, HistGradientBoosting, + latitud/longitud: {m2:.3f} ± {s2:.3f}")

Con solo las cuatro variables que la ecuación 1 declara, un modelo no lineal apenas explica el 18 % de
la varianza del IAS. Añadir dos coordenadas geográficas, que no aparecen en la fórmula, lo sube a 91
%. La mayor parte de lo que el modelo aprende no viene de la topografía declarada.

In [ ]:
def leave_region_out_r2(features):
    sub = df_clean.dropna(subset=features + ["solar_aptitude", "macro_region"])
    out = {}
    for region in sub["macro_region"].unique():
        train = sub[sub["macro_region"] != region]
        test = sub[sub["macro_region"] == region]
        model = HistGradientBoostingRegressor(random_state=SEED).fit(
            train[features], train["solar_aptitude"])
        pred = model.predict(test[features])
        out[region] = r2_score(test["solar_aptitude"], pred)
    return out

all_numeric = ["latitude", "longitude", "elevation", "slope", "curvature",
               "aspect_sin", "aspect_cos", "ambient_temperature", "humidity", "wind_speed"]

print("R² dejando una macro-región fuera del entrenamiento:")
for region, r2 in leave_region_out_r2(all_numeric).items():
    print(f"  Prueba en {region:28s} R² = {r2:+.3f}")

Un R² negativo significa que el modelo predice peor que simplemente usar el promedio de la región.
Entrenado en dos regiones y probado en la tercera, con diez variables numéricas disponibles, el
modelo pierde por completo. Si el IAS reflejara una relación física estable entre terreno y aptitud,
esa relación debería transferirse entre continentes. No se transfiere.

La explicación más simple, consistente con lo que describe el propio paper en su sección 2.4.3, es que
el DEM se procesó por continente y cada lote se normalizó de manera independiente. El repositorio
oficial del dataset (`cimejia/solarPV`) ofrece evidencia adicional en esa dirección: en una versión
anterior del archivo, publicada antes de incorporar Europa, la clase Baja casi no existía, con apenas
204 filas frente a las 1,892 actuales. La gran mayoría de la clase Baja actual entró en el dataset
junto con el lote europeo. Esto no invalida el dataset para este proyecto, pero sí cambia lo que se
puede afirmar con él: el IAS funciona razonablemente bien para comparar plantas dentro de una misma
región, y no es fiable para comparar entre regiones sin controlar ese efecto de lote. La validación
cruzada tiene que diseñarse teniendo esto en cuenta, que es el tema de la siguiente sección.

## 10. Esquema de validación cruzada espacial

Ninguna de las dos entregas anteriores usó partición espacial, pese a que la primera la declaraba
como palabra clave sin implementarla. Dado el hallazgo de la sección 9, una partición aleatoria deja
que coordenadas casi idénticas caigan una en entrenamiento y otra en prueba, lo que permite
interpolación espacial pura y sobreestima el desempeño real del modelo. Roberts et al. (2017)
documentan este problema en detalle para datos con estructura espacial, temporal o jerárquica, y
recomiendan particionar por bloques en vez de al azar quede como estándar en ese tipo de datos. El
notebook de referencia de los propios autores del dataset sigue la misma lógica, con `GroupKFold`
sobre bloques de 0.5 grados.

In [ ]:
df_clean["block_lat"] = (df_clean["latitude"] // 5) * 5
df_clean["block_lon"] = (df_clean["longitude"] // 5) * 5
df_clean["spatial_block"] = (df_clean["block_lat"].astype(str) + "_" +
                              df_clean["block_lon"].astype(str))

print(f"Bloques espaciales de 5°x5°: {df_clean['spatial_block'].nunique()}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
y = df_clean["solar_aptittude_class"]
groups = df_clean["spatial_block"]

for fold, (tr_idx, te_idx) in enumerate(sgkf.split(df_clean, y, groups)):
    n_baja_test = (y.iloc[te_idx] == "Baja").sum()
    print(f"  Fold {fold}: n_test={len(te_idx):6,d}  Baja_test={n_baja_test:4d}  "
          f"({n_baja_test / len(te_idx) * 100:.2f}%)")

Usamos 5 grados por lado en vez de los 0.5 del notebook de referencia porque ahí trabajan sobre un
subconjunto de Sudamérica con 7,000 filas, mientras que aquí el dataset es global y tiene casi 58,000.
Con bloques tan finos como 0.5 grados, el número de grupos distintos sería enorme y muchos quedarían
con una sola planta, lo que rompe la lógica de `GroupKFold`. Con 496 bloques de 5 grados, los cinco
pliegues quedan con un porcentaje de Baja parecido entre sí, entre 2.9 % y 3.8 %, así que ningún
pliegue se queda sin ejemplos de la clase minoritaria. Si en el cuaderno de modelos algún pliegue
resulta demasiado desbalanceado para un modelo específico, la alternativa es reducir el bloque a 2.5
grados o agrupar por país en su lugar.

## 11. Conjunto de predictoras final

La regla de inclusión es simple: una variable entra si está disponible antes de invertir en una
coordenada cualquiera, si no es una copia determinista de otra variable ya incluida, y si no arrastra
un centinela sin tratar. Por eso quedan fuera `area` y `size`, que son posteriores a la construcción de
la planta y traen el problema de D5 a D7; los campos `*_type` y `dt_wind`, que son binnings de D12; y
`capacity` junto con `operational_status`, que solo existen después de decidir invertir.

In [ ]:
predictors_final = [
    "latitude", "longitude", "elevation", "slope", "curvature",
    "aspect_sin", "aspect_cos", "is_flat", "wind_sin", "wind_cos",
    "log_dist_to_road", "ambient_temperature", "humidity", "wind_speed", "ghi",
]
print(f"{len(predictors_final)} predictoras finales: {predictors_final}")

X = df_clean[predictors_final]
y_clf = df_clean["solar_aptittude_class"]
y_reg = df_clean["solar_aptitude"]

print(f"\nX: {X.shape}")
na_counts = X.isna().sum()
print("Nulos por columna:")
print(na_counts[na_counts > 0] if na_counts.sum() else "  (ninguno)")

Quedan 63 valores nulos, todos en `log_dist_to_road`, que vienen de las distancias mayores a 100 km
que tratamos como error en la sección 5. Son menos del 0.11 % de las filas, así que una imputación
simple por mediana dentro de cada pliegue de entrenamiento basta para el cuaderno de modelos, sin
necesidad de descartar esas filas.

In [ ]:
export_cols = predictors_final + [
    "solar_aptitude", "solar_aptitude_rounded", "solar_aptittude_class",
    "spatial_block", "macro_region", "country", "code",
]
df_export = df_clean[export_cols].copy()
df_export.to_csv("dataset_eda_corregido.csv", index=False)
print(f"Exportado: dataset_eda_corregido.csv "
      f"({df_export.shape[0]:,} filas, {df_export.shape[1]} columnas)")

## 12. Conclusiones

El target que vamos a usar en el cuaderno de modelos no cambia respecto al planteado en la Entrega 2:
`solar_aptittude_class` para clasificación y `solar_aptitude` para regresión, la misma variable en dos
formas. Lo que sí cambia es cómo la describimos y cómo la validamos, y eso es gran parte de lo que
este cuaderno corrige.

Sobre calidad de datos, documentamos y tratamos doce problemas que la Entrega 1 no reportó o reportó
de forma incompleta. El más importante para el curso es D8: la afirmación de que el dataset no tenía
duplicados era falsa, con más de 1,500 duplicados de atributos y casi 10,000 filas que comparten
coordenada con alguna otra. D1, D2 y D5 a D7 muestran que varios huecos del dataset no son aleatorios,
sino artefactos del método de construcción: cobertura incompleta de DEM y de GSA, y un procedimiento
de unión punto-polígono que falla sistemáticamente al calcular `area`.

El hallazgo con más peso para todo lo que viene después es el de la sección 9. El IAS correlaciona
casi diez veces más fuerte con la longitud que con la pendiente, pese a que la pendiente pesa 40 % en
la fórmula del índice y la longitud no entra en absoluto. Un modelo entrenado en dos macro-regiones
pierde por completo al predecir la tercera. Esto no significa que el dataset esté mal construido para
los fines de este curso: significa que hay que ser honestos sobre qué mide el índice. Mide, sobre
todo, en qué lote de procesamiento cayó cada planta, con una relación topográfica real pero secundaria
dentro de cada región. Eso convierte la validación cruzada espacial de la sección 10, con bloques de 5
grados y `StratifiedGroupKFold`, en un requisito y no en una opción entre varias.

De cara al cuaderno de modelos, quedan tres implicaciones directas. Primera, cualquier métrica
reportada con partición aleatoria va a verse mejor de lo que el modelo realmente generaliza, así que el
bucle externo de la validación anidada tiene que usar `spatial_block` como grupo, nunca una partición
al azar. Segunda, el desbalance de clases (75 % Alta, 22 % Media, 3 % Baja) sigue intacto después de la
limpieza, así que las cuatro técnicas de balanceo que exige la guía, SMOTE (Chawla et al., 2002),
ADASYN (He et al., 2008) y `class_weight='balanced'`, tienen un motivo real detrás y no son solo un
ejercicio obligatorio. Tercera, al interpretar SHAP sobre el mejor modelo, conviene anticipar que
`latitude` y `longitude` van a dominar la importancia de variables, seguidas de `slope` con una
relación negativa dentro de cada región y de `log_dist_to_road`. Si eso se confirma, hay que
reportarlo como el hallazgo central del proyecto, no esconderlo detrás de una métrica de desempeño
alta.

El siguiente cuaderno, `03_modelos_clasificacion_regresion.ipynb`, parte de `dataset_eda_corregido.csv`
y entrena los 112 modelos de clasificación y 28 de regresión que exige la guía de la Entrega 3,
usando `spatial_block` como grupo en el bucle externo de la validación anidada.

## Apéndice: índice de variables

Columnas del dataset original (Tabla 2 del paper fuente), con la decisión que tomamos para cada una
y por qué.

In [ ]:
variable_index = pd.DataFrame([
    ("OBJECTID",              "Identificación", "entero",  "-",        "Identificador interno del SIG",                          "Excluida: identificador, no predictora"),
    ("code",                  "Identificación", "texto",   "-",        "Código alfanumérico de la planta",                       "Excluida: identificador"),
    ("plant_name",            "Identificación", "texto",   "-",        "Nombre de la planta (GEM / GSA)",                        "Excluida: identificador"),
    ("country",                "Geográfica",     "texto",   "-",        "País de la planta",                                      "Excluida como predictora directa; usada como grupo alternativo de CV"),
    ("latitude",               "Geográfica",     "numérica","grados",   "Coordenada de latitud",                                  "Incluida"),
    ("longitude",              "Geográfica",     "numérica","grados",   "Coordenada de longitud",                                 "Incluida: domina la correlación con el target, ver sección 9"),
    ("elevation",              "Topográfica",    "numérica","m",        "Elevación sobre el nivel del mar",                       "Incluida"),
    ("slope",                  "Topográfica",    "numérica","grados",   "Inclinación del terreno",                                "Incluida: componente de la ecuación 1 (peso 0.40)"),
    ("aspect",                 "Topográfica",    "numérica","grados",   "Orientación de la pendiente, -1 si es plana",            "Transformada a aspect_sin, aspect_cos, is_flat (D3)"),
    ("curvature",              "Topográfica",    "numérica","-",        "Forma del terreno, cóncavo o convexo",                   "Incluida: componente de la ecuación 1 (peso 0.15)"),
    ("slope_type",             "Topográfica",    "categórica","-",      "Clasificación de slope",                                 "Excluida: binning determinista de slope (D12)"),
    ("aspect_type",            "Topográfica",    "categórica","-",      "Clasificación de aspect",                                "Excluida: binning determinista de aspect (D12)"),
    ("curvature_type",         "Topográfica",    "categórica","-",      "Clasificación de curvature",                             "Excluida: binning determinista de curvature (D12)"),
    ("dist_to_road",           "Logística",      "numérica","m",        "Distancia a la carretera más cercana",                   "Transformada a log_dist_to_road, >100 km tratado como error (D4)"),
    ("area",                   "Logística",      "numérica","m2",       "Área del polígono de la planta en OSM",                  "Excluida: posterior a la construcción, densidad inconsistente (D5-D7)"),
    ("size",                   "Logística",      "categórica","-",      "Clasificación de area",                                  "Excluida: hereda el problema de area, confunde NaN con 'Small' (D6)"),
    ("ambient_temperature",    "Climática",      "numérica","°C",       "Temperatura media anual",                                "Incluida"),
    ("humidity",               "Climática",      "numérica","%",        "Humedad relativa media anual",                           "Incluida"),
    ("wind_speed",             "Climática",      "numérica","m/s",      "Magnitud del viento vectorial medio",                    "Incluida, con la salvedad de D11"),
    ("wind_direction",         "Climática",      "numérica","grados",   "Dirección predominante del viento",                      "Transformada a wind_sin, wind_cos"),
    ("dt_wind",                "Climática",      "categórica","-",      "Clasificación de wind_direction",                        "Excluida: binning determinista (D12)"),
    ("operational_status",     "Potencia",       "categórica","-",      "Fase del proyecto (operando, cancelado, etc.)",          "Excluida: posterior a la decisión de invertir"),
    ("solar_aptitude",         "Potencia",       "numérica","-",        "Índice de aptitud solar (IAS), ecuación 1",              "Target de regresión"),
    ("solar_aptitude_rounded", "Potencia",       "entero",  "-",        "IAS redondeado a un dígito",                             "Excluida: inconsistencias de redondeo propias (D10)"),
    ("solar_aptittude_class",  "Potencia",       "categórica","-",      "IAS discretizado en Baja, Media, Alta",                  "Target de clasificación"),
    ("optimal_tilt",           "Potencia",       "numérica","grados",   "Ángulo óptimo de inclinación del panel",                 "Disponible antes de invertir; se deja fuera del conjunto base por redundancia con latitud, decisión abierta del equipo"),
    ("ghi",                    "Potencia",       "numérica","kWh/m2/día","Irradiación global en plano inclinado (GTI real)",      "Incluida"),
    ("capacity",               "Potencia",       "numérica","MW",       "Potencia instalada",                                     "Excluida: posterior a la decisión de invertir"),
    ("pv_potential",           "Potencia",       "numérica","kWh/kWp/día","Energía generada por kWp instalado",                   "Disponible antes de invertir; se deja fuera del conjunto base por redundancia con ghi, decisión abierta del equipo"),
], columns=["variable", "categoría", "tipo", "unidad", "descripción", "decisión en este EDA"])

pd.set_option("display.max_colwidth", 90)
variable_index

`optimal_tilt` y `pv_potential` están disponibles antes de construir la planta, así que no violan la
regla de fuga temporal, pero las dejamos fuera del conjunto base porque están muy correlacionadas con
variables ya incluidas (`optimal_tilt` con la latitud, `pv_potential` con `ghi`). Si el cuaderno de
modelos necesita una variante con más variables, esta tabla ya documenta la razón para añadirlas o no.

## Referencias

Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J., Garcia-Rodriguez, J., Tarco, B. F., y
Santamaria, K. (2026). *Global dataset of solar power plants: multidimensional integration and
analysis*. arXiv:2603.20601.

FAO (1976). *A framework for land evaluation*. Food and Agriculture Organization of the United
Nations.

Wilson, J. P., y Gallant, J. C. (2000). *Terrain analysis: principles and applications*. John Wiley
and Sons.

Saaty, T. L. (1980). *The analytic hierarchy process*. McGraw-Hill.

Roberts, D. R., Bahn, V., Ciuti, S., Boyce, M. S., Elith, J., Guillera-Arroita, G., Hauenstein, S.,
Lahoz-Monfort, J. J., Schröder, B., Thuiller, W., Warton, D. I., Wintle, B. A., Hartig, F., y
Dormann, C. F. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or
phylogenetic structure. *Ecography*, 40(8), 913 a 929.

Fisher, N. I. (1993). *Statistical analysis of circular data*. Cambridge University Press.

Hollander, M., Wolfe, D. A., y Chicken, E. (2014). *Nonparametric statistical methods* (3.a ed.).
Wiley.

O'Brien, R. M. (2007). A caution regarding rules of thumb for variance inflation factors. *Quality
and Quantity*, 41, 673 a 690.

Chawla, N. V., Bowyer, K. W., Hall, L. O., y Kegelmeyer, W. P. (2002). SMOTE: synthetic minority
over-sampling technique. *Journal of Artificial Intelligence Research*, 16, 321 a 357.

He, H., Bai, Y., Garcia, E. A., y Li, S. (2008). ADASYN: adaptive synthetic sampling approach for
imbalanced learning. *Proceedings of the IEEE International Joint Conference on Neural Networks*,
1322 a 1328.

King, G., y Zeng, L. (2001). Logistic regression in rare events data. *Political Analysis*, 9(2),
137 a 163.

# Resumen final

Este apartado es el mismo en los cuatro cuadernos del proyecto. Resume todo el trabajo, desde la recolección de los datos hasta los benchmarks, con las métricas,
las decisiones y la razón de cada una. Las cifras salen de la ejecución de los cuadernos el 2 de octubre de 2026.

| Cuaderno | Qué hace |
|---|---|
| `EDA_Corregido(Entregable 3).ipynb` | Limpia y explora la base mundial y produce `dataset_eda_corregido.csv` |
| `Benchmark_Modelos_Base.ipynb` | Modelos base del curso sobre la base mundial |
| `Clima_a_Generacion_Colombia.ipynb` | Construye la base de generación real de Colombia, la explora y prueba modelos preliminares |
| `Benchmark_Colombia.ipynb` | Modelos base del curso sobre la base de Colombia |

## R0. Guion de la presentación, diapositiva por diapositiva

Cada diapositiva dice qué mostrar, qué decir y el número que hay que citar. El detalle y las referencias de cada punto están en las secciones R1 a R15 de este resumen.

**Diapositiva 1. Título y pregunta.**
Qué decir: queríamos predecir qué tan apto es un lugar para una planta solar, a partir del clima, para que sirviera en cualquier parte. Trabajamos con un dataset
mundial publicado en 2026 (Mantilla-Guerra et al., 2026) de 58,978 plantas solares.

**Diapositiva 2. Los datos.**
Mostrar: la tabla de fuentes de R2.
Qué decir: el dataset une cinco fuentes. Global Energy Monitor da las plantas, Global Solar Atlas el potencial solar, ERA5 y NASA POWER el clima, los modelos
de elevación el terreno y OpenStreetMap las carreteras. Son 29 columnas. La variable a predecir es un índice de aptitud solar de 0 a 1 y su versión en tres clases:
Baja, Media y Alta.

**Diapositiva 3. Qué mide de verdad el índice (el primer hallazgo).**
Mostrar: la ecuación del índice de R1.
Qué decir: el paper calcula el índice solo con pendiente, orientación, sombreado y curvatura. El clima no entra. Por eso, aunque quisiéramos, el clima no puede
predecir este índice. Esto cambió el proyecto.

**Diapositiva 4. Pipeline del EDA.**
Mostrar: `fig_pipeline_eda_mundial.png`.
Qué decir: recorrer las cajas. De 58,978 plantas a 57,976 tras quitar filas rotas y duplicados. Se crean variables nuevas, se eligen 15 predictoras y se asigna
cada planta a un bloque del mapa de 5° × 5°.

**Diapositiva 5. Problemas de calidad de datos.**
Mostrar: la tabla de problemas de R3.
Qué decir: 64 plantas sin datos de terreno, todas en la clase Baja. 1,598 duplicados. Distancias a carretera de hasta 2,296 km, que son imposibles. Un 27.7 % de
áreas vacías. Para cada una, la tabla dice qué decisión se tomó y por qué. Los ángulos se pasan a seno y coseno porque 359° y 1° son vecinos.

**Diapositiva 6. El hallazgo central del EDA.**
Mostrar: `fig_mapa_clases_e3.png` o `fig_correlacion_e3.png` del EDA.
Qué decir:
- El índice casi no se relaciona con la pendiente (Spearman -0.07), con la que supuestamente se calcula, y sí con la longitud (+0.58).
- Con solo terreno, un modelo explica R² de 0.18. Con latitud y longitud, 0.91.
- El 94.5 % de la clase Baja viene de un lote de plantas europeas añadido después.

Conclusión: el índice mide sobre todo la región y el lote de procesamiento, no el terreno.

**Diapositiva 7. Por qué validación espacial.**
Mostrar: `fig_pipeline_benchmark_mundial.png`.
Qué decir: si partimos los datos al azar, plantas vecinas quedan una en entrenamiento y otra en prueba, y el modelo "hace trampa" por vecindad (Roberts et al., 2017).
Por eso partimos por bloques de 5° × 5°: un bloque entero va a entrenamiento o a prueba. Además, todo lo que se aprende de los datos (la mediana para rellenar
vacíos, la escala) se ajusta solo con el entrenamiento, dentro de un `Pipeline`, para no filtrar información de la prueba (Kaufman et al., 2012).

**Diapositiva 8. Modelos y métricas.**
Mostrar: las tablas de R6 y R7.
Qué decir: usamos los modelos base del curso. En clasificación, regresión logística, Naive Bayes, KNN y SVM. En regresión, Ridge, Lasso, KNN y SVR. La métrica principal
de clasificación es F1 macro, porque la clase Baja es solo el 3 % y la exactitud premiaría ignorarla. También reportamos AUC, que mide si las probabilidades
ordenan bien sin depender de un umbral, y kappa ponderado, porque las clases tienen orden. En regresión, R², RMSE y MAE.

**Diapositiva 9. hyperparameter tuning sin trampa.**
Mostrar: el esquema de validación anidada de R5.
Qué decir: si elegimos los hiperparámetros mirando los mismos datos con los que reportamos el resultado, el resultado sale inflado (Varma y Simon, 2006).
Por eso usamos validación anidada: un bucle interno elige los hiperparámetros, y otro externo, que nunca participó en la elección, mide.

**Diapositiva 10. Resultados de la base mundial.**
Mostrar: `fig_resumen_comparativa.png` (fila de arriba) y `fig_benchmark_roc_confusion.png`.
Qué decir: el SVM con kernel RBF es el mejor clasificador (F1 macro 0.82, AUC 0.96) y el SVR RBF el mejor en regresión (R² 0.81). KNN va detrás. Los lineales
quedan en F1 0.60 y R² 0.48. La referencia, que siempre predice la clase más común, da F1 0.29.

**Diapositiva 11. Pero, ¿qué aprendieron?**
Mostrar: `fig_benchmark_modelos.png`.
Qué decir:
- Al quitar latitud y longitud, KNN baja de 0.79 a 0.48 en R². Aprende dónde está la planta.
- Con partición aleatoria los resultados salen más altos que con bloques.
- Dejando un continente entero fuera, todos los modelos dan R² negativo, hasta la referencia.

El modelo no sirve fuera de las regiones que vio, y eso confirma el hallazgo del EDA.

**Diapositiva 12. Por qué los lineales quedan bajos.**
Mostrar: la interpretación de R11.
Qué decir: un modelo lineal ajusta un solo plano para todo el mundo, y el índice tiene un nivel distinto en cada región. A eso se le llama no estacionariedad
espacial (Brunsdon et al., 1996). KNN y SVM con kernel ajustan vecindarios locales. Para mejorar se podrían usar bosques aleatorios o XGBoost, modelos espaciales o
stacking. Pero subir el puntaje no hace que el índice mida aptitud.

**Diapositiva 13. La segunda línea: generación real en Colombia.**
Mostrar: `fig_pipeline_clima.png`.
Qué decir: para volver a la idea original del clima, construimos una base propia:
- Generación real hora por hora de 16 plantas colombianas, desde la API pública de XM.
- Clima de cada sitio desde Open-Meteo.
- Coordenadas verificadas con prensa y con las fichas de Global Energy Monitor.

El resultado son 8,589 días planta, de enero de 2024 a febrero de 2026, sin valores vacíos.

**Diapositiva 14. EDA de la base de Colombia.**
Mostrar: `fig_eda_base_colombia.png` y `fig_fc_vs_radiacion.png`.
Qué decir: lo que predecimos es el factor de capacidad, la energía del día dividida entre la máxima posible. El 84 % de su variación es de un día a otro dentro de
cada planta. La radiación es la variable que más se relaciona con él, y nubosidad, temperatura y humedad están muy correlacionadas con ella. Falta la tecnología
de cada planta (si tiene seguidor solar), que es pendiente.

**Diapositiva 15. Resultados en Colombia.**
Mostrar: `fig_resumen_comparativa.png` (fila de abajo) y `fig_benchmark_colombia.png`.
Qué decir:
- Aquí ganan los modelos lineales. Lasso explica R² de 0.47 del sube y baja diario, y el error baja del 20 % al 14 % del factor de capacidad medio.
- Clasificar el día como bajo, medio o alto: la regresión logística acierta 57 de cada 100, con AUC 0.77. El azar daría 33.
- Cada kWh/m² de radiación suma unos 4.7 puntos de factor de capacidad, como predice la física.
- La partición aleatoria vuelve a inflar los resultados.

**Diapositiva 16. Conclusiones.**
Mostrar: R12.
Qué decir: el índice del dataset no mide aptitud desde el clima y no se transfiere entre regiones. Con datos reales, el clima sí explica cerca de la mitad de la
variación diaria de la generación, y casi todo lo aporta la radiación.

**Diapositiva 17. Limitaciones y siguientes pasos.**
Mostrar: R13.
Qué decir: falta la tecnología de cada planta, radiación medida en lugar de modelada, datos por hora, y los modelos de la entrega siguiente (árboles, XGBoost, balanceo con SMOTE).

Preguntas probables y respuesta corta:
- ¿Por qué no partición aleatoria? Porque plantas vecinas se parecen y el resultado sale optimista. Lo mostramos con números en la diapositiva 11.
- ¿Por qué F1 macro y no exactitud? Porque con 75 % de clase Alta, predecir siempre Alta da 75 % de exactitud y no sirve para nada.
- ¿Por qué tan pocas plantas en Colombia? Porque XM no publica coordenadas, y solo usamos plantas cuya ubicación pudimos verificar con fuentes.
- ¿Qué es el AUC? La probabilidad de que el modelo le dé más puntaje a un ejemplo de la clase correcta que a uno de otra clase. 0.5 es azar y 1 es perfecto.

## R1. La pregunta y por qué cambió

La propuesta inicial era predecir la aptitud solar de un sitio a partir del clima, para tener algo útil en cualquier lugar. El dataset del proyecto
(Mantilla-Guerra et al., 2026) trae un índice de aptitud solar (`solar_aptitude`) calculado con la ecuación 1 del paper:
`IAS = 0.40 · pendiente + 0.25 · orientación + 0.20 · sombreado + 0.15 · curvatura`. El clima no entra en esa fórmula, así que no puede predecirse desde el clima.
El EDA confirmó además que el índice depende sobre todo de la región y del lote de procesamiento.

Por eso el trabajo tiene dos líneas:
1. Modelar el índice del dataset y mostrar, con validación espacial, por qué no se transfiere entre regiones.
2. Construir una base propia con la generación real de plantas colombianas y medir cuánto la explica el clima, que es la idea original.

## R2. Recolección de datos

| Fuente | Qué aporta | Acceso |
|---|---|---|
| Dataset de Mantilla-Guerra et al. (2026) | 58,978 plantas FV del mundo, 29 columnas: ubicación, terreno, clima anual, distancia a carretera, potencial FV y el índice | CSV del repositorio de los autores |
| Global Energy Monitor, Global Solar Power Tracker (release de febrero de 2026) | Base de plantas del dataset y fichas públicas con coordenadas | Web de GEM |
| XM, operador del mercado eléctrico colombiano | Generación horaria por planta y capacidad efectiva neta diaria, enero de 2024 a febrero de 2026 | API pública, sin clave |
| Open-Meteo, API histórica (Zippenfenig, 2023) | Radiación, nubosidad, temperatura, humedad y viento por hora, con los productos ERA5 (Hersbach et al., 2020) y `best_match` | API pública, sin clave |

![Pipeline del EDA](fig_pipeline_eda_mundial.png)

![Pipeline de la base de Colombia](fig_pipeline_clima.png)

## R3. EDA y limpieza de la base mundial

Problemas encontrados y cómo se resolvieron:

| Problema | Cifra | Decisión |
|---|---|---|
| Filas sin cobertura de terreno (DEM) | 64 filas, todas en clase Baja (3.4 % de esa clase) | Eliminar |
| Filas sin datos de clima o potencial | 3 filas, más 2 con viento en 0 | Eliminar |
| Duplicados de atributos | 1,598 filas iguales salvo el identificador | Eliminar |
| Orientación con valor -1 en terreno plano | 1,105 filas | Seno y coseno igual a 0 más un indicador `is_flat` |
| Distancia a carretera imposible | 65 filas a más de 100 km, máximo 2,296 km | Pasar a vacío y usar logaritmo |
| Área de la planta | 16,329 vacíos (27.7 %), densidades físicamente imposibles | Excluir |
| Variables que son agrupaciones de otras (`*_type`, `size`, `dt_wind`) | Rangos sin solapamiento | Excluir |

Resultado: 57,976 plantas y 15 predictoras (`dataset_eda_corregido.csv`, 57,976 × 22).

Decisiones y su respaldo:
- Los ángulos (orientación y dirección del viento) se pasan a seno y coseno, porque 359° y 1° son vecinos (Fisher, 1993).
- Se usa correlación de Spearman en lugar de Pearson por las colas pesadas y relaciones no lineales (Hollander et al., 2015).
- Los umbrales de VIF se interpretan con cautela (O'Brien, 2007).
- Se excluyen variables posteriores a la decisión de invertir (capacidad, estado operativo) para no meter información del futuro en el modelo (Kaufman et al., 2012).

Hallazgo central:
- El índice casi no se relaciona con las variables de terreno con las que se calcula: Spearman de -0.066 con la pendiente. Sí se relaciona con la longitud (+0.583).
- Un modelo con solo pendiente, orientación y curvatura explica R² de 0.18. Al añadir latitud y longitud sube a 0.91.
- Dejando fuera una macrorregión, el R² es negativo (-7.3 en Asia y Oceanía, -2.3 en América, -0.1 en Europa, África y Medio Oriente).
- El 94.5 % de la clase Baja viene de un lote europeo que se añadió en la segunda versión del dataset.

La consecuencia es que la validación debe ser espacial. Con partición aleatoria, plantas vecinas quedan a los dos lados y el resultado sale optimista
(Roberts et al., 2017; Ploton et al., 2020). Se definieron 496 bloques de 5° × 5°. En los 5 pliegues, la clase Baja queda entre 2.9 % y 3.8 %.

## R4. Construcción de la base de Colombia

1. Se listaron los 2,213 recursos solares de XM y se quedaron los 27 con al menos 1 MW de capacidad.
2. Se calculó el factor de capacidad diario: `FC = kWh del día / (capacidad efectiva neta × 24 h)`, con la capacidad de ese mismo día.
3. Coordenadas: XM no las publica. Nueve plantas se emparejaron con el dataset del paper y siete con la ficha pública de GEM. Cada emparejamiento lleva
   su motivo y una fuente de prensa o del desarrollador. Por ejemplo, el parque Fundación está en Pivijay (Magdalena). Se descartaron Palmira II, porque
   su ubicación no se confirmó, y Shangri-La, porque sus coordenadas en GEM contradicen su propia ficha.
4. Clima por hora de Open-Meteo en cada coordenada, con dos productos (`era5` y `best_match`). Se resumió por día: radiación diaria, nubosidad media en
   horas de luz, temperatura, humedad y viento. Las horas de XM y de Open-Meteo coinciden sin desfase, y el resumen diario evita problemas de alineación.
5. Filtros con criterios que no miran la variable que se predice: se excluyen los primeros 60 días de las plantas con arranque observado y las plantas con menos de 100 días útiles.

Resultado: `tabla_diaria_colombia.csv`, con 8,589 filas planta-día, 23 columnas, 16 plantas, 6 zonas de cercanía y ningún valor vacío fuera de `fc_ayer`.
Los días sin generación en XM van de 0 a 24 por planta.

EDA de la base:
- El 84 % de la varianza del FC es de un día a otro dentro de cada planta. El 16 % es diferencia entre plantas.
- Dentro de la planta, la radiación diaria explica cerca de la mitad del sube y baja del FC. Cada kWh/m² suma unos 4.7 puntos de FC, casi la proporción física simple.
- Nubosidad, temperatura y humedad están muy correlacionadas con la radiación.
- Pendiente: la tecnología de cada planta (seguidor o fija, potencia DC frente a AC). Ninguna de las 16 la tiene confirmada.

![EDA de la base de Colombia](fig_eda_base_colombia.png)

![FC contra radiación por planta](fig_fc_vs_radiacion.png)

## R5. Diseño experimental común

La partición va siempre primero, y todo lo que se aprende de los datos se ajusta solo con el entrenamiento de cada partición (Kaufman et al., 2012):

| Decisión | Base mundial | Base de Colombia | Respaldo |
|---|---|---|---|
| Unidad de partición | Bloques espaciales de 5° × 5°, 5 pliegues | Zona de plantas a menos de 100 km, más división en el tiempo (pasado para entrenar, futuro para probar) | Roberts et al. (2017); Ploton et al. (2020); Meyer y Pebesma (2021) |
| Imputación y escalado | Dentro de un `Pipeline` de scikit-learn | Dentro de un `Pipeline` | Pedregosa et al. (2011); Kaufman et al. (2012) |
| Nivel de cada planta y cortes de clase | No aplica | Calculados solo con el pasado de cada planta | Kaufman et al. (2012) |
| hyperparameter tuning | Validación anidada: 5 outer folds y 3 internos, todos por bloques | Validación anidada: 6 zonas externas, y en el interno se deja fuera una zona de entrenamiento a la vez | Varma y Simon (2006); Cawley y Talbot (2010) |
| Búsqueda | Grillas pequeñas en escala logarítmica para C y alpha | Igual | Bergstra y Bengio (2012) |
| Probabilidades del SVM lineal | Calibradas con `CalibratedClassifierCV` dentro del entrenamiento | Igual | Niculescu-Mizil y Caruana (2005) |
| SVM con kernel en la base mundial | Submuestra de 8,000 filas por pliegue, porque su costo crece entre cuadrático y cúbico con las filas | No hace falta | Chang y Lin (2011) |

Por qué validación anidada: si se eligen los hiperparámetros con los mismos datos con los que se reporta el resultado, la estimación sale optimista
(Varma y Simon, 2006; Cawley y Talbot, 2010). El bucle interno elige y el externo solo mide.

## R6. Modelos y por qué cada uno

| Modelo | Tarea | Por qué se incluye | Referencia |
|---|---|---|---|
| Referencia (clase mayoritaria, media o nivel de la planta) | Ambas | Mide el piso: cualquier modelo útil debe superarlo | |
| Regresión logística (y versión balanceada) | Clasificación | Lineal, interpretable. La versión con `class_weight="balanced"` compensa la clase Baja | Hastie et al. (2009); He y Garcia (2009) |
| Naive Bayes gaussiano | Clasificación | Modelo probabilístico simple y rápido, línea base del curso | Hastie et al. (2009) |
| KNN | Ambas | No supone forma de la relación. Sensible a la escala, por eso se estandariza | Cover y Hart (1967) |
| SVM lineal y SVM RBF | Clasificación | Margen máximo. El kernel RBF captura relaciones no lineales | Cortes y Vapnik (1995); Chang y Lin (2011) |
| Ridge | Regresión | Lineal con penalización L2, estable con variables correlacionadas | Hoerl y Kennard (1970) |
| Lasso | Regresión | Penalización L1, puede anular variables que sobran | Tibshirani (1996) |
| SVR lineal y SVR RBF | Regresión | Versión de regresión del SVM | Smola y Schölkopf (2004) |
| HistGradientBoosting | Regresión (Colombia, preliminar) | Control flexible basado en árboles | Friedman (2001) |

## R7. Métricas y por qué

| Métrica | Qué mide | Por qué se usa | Referencia |
|---|---|---|---|
| F1 macro | Promedio del F1 de cada clase | Métrica principal en clasificación. Con la clase Baja en 3 %, la exactitud premiaría ignorarla | He y Garcia (2009); Sokolova y Lapalme (2009) |
| Exactitud y exactitud balanceada | Aciertos totales y promedio del recall por clase | La balanceada no depende de la proporción de clases | Sokolova y Lapalme (2009) |
| Precisión y recall macro | Cuánto se acierta al predecir una clase y cuánto se recupera de ella | Separan los dos tipos de error | Sokolova y Lapalme (2009) |
| F1 de la clase Baja | F1 solo de la minoritaria | Es la clase difícil y la más afectada por el lote europeo | He y Garcia (2009) |
| AUC ROC macro uno contra el resto | Qué tan bien ordenan las probabilidades a cada clase frente a las demás, sin umbral | Complementa al F1, que depende del umbral | Fawcett (2006); Hand y Till (2001) |
| Kappa ponderado cuadrático | Acuerdo que castiga más los errores entre clases lejanas | Las clases tienen orden: confundir Baja con Alta es peor que con Media | Cohen (1968) |
| R² | Proporción de la varianza explicada | Métrica principal en regresión. En Colombia se calcula dentro de cada planta | Hastie et al. (2009) |
| RMSE y MAE | Error en las unidades de la variable. RMSE castiga más los errores grandes | Dan la magnitud del error | Hastie et al. (2009) |

## R8. Resultados de la base mundial

Con hiperparámetros por defecto, bloques de 5° y con latitud y longitud (clasificación):

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | F1 Baja | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|---|
| Referencia | 0.285 | 0.748 | 0.333 | 0.249 | 0.333 | 0.000 | 0.500 | 0.000 |
| Regresión logística | 0.599 | 0.768 | 0.559 | 0.695 | 0.559 | 0.523 | 0.865 | 0.367 |
| Regresión logística balanceada | 0.629 | 0.719 | 0.798 | 0.587 | 0.798 | 0.507 | 0.873 | 0.481 |
| Naive Bayes | 0.448 | 0.574 | 0.657 | 0.515 | 0.657 | 0.169 | 0.768 | 0.261 |
| KNN | 0.787 | 0.897 | 0.742 | 0.859 | 0.742 | 0.658 | 0.951 | 0.737 |
| SVM lineal (calibrado) | 0.516 | 0.753 | 0.479 | 0.668 | 0.479 | 0.358 | 0.874 | 0.246 |
| SVM RBF | 0.812 | 0.896 | 0.772 | 0.869 | 0.772 | 0.747 | 0.958 | 0.751 |

Regresión, mismo escenario:

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| Referencia | -0.002 | 0.134 | 0.103 |
| Ridge | 0.479 | 0.096 | 0.069 |
| Lasso | 0.478 | 0.096 | 0.069 |
| KNN | 0.790 | 0.061 | 0.037 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| SVR RBF | 0.734 | 0.069 | 0.053 |

Los contrastes (F1 macro y R², por defecto):

| Modelo | F1 bloques con lat/lon | F1 bloques sin lat/lon | F1 aleatoria sin lat/lon | R² bloques con lat/lon | R² bloques sin lat/lon | R² aleatoria sin lat/lon |
|---|---|---|---|---|---|---|
| KNN | 0.787 | 0.658 | 0.730 | 0.790 | 0.480 | 0.656 |
| SVM / SVR RBF | 0.812 | 0.617 | 0.680 | 0.734 | 0.427 | 0.560 |
| Regresión logística / Ridge | 0.599 | 0.377 | 0.381 | 0.479 | 0.195 | 0.220 |

Dejando fuera una macrorregión completa, todos los modelos de regresión dan R² negativo, incluida la referencia.

Con hiperparámetros ajustados por validación anidada (5 outer folds y 3 internos, todos por bloques; bloques de 5° y con latitud y longitud):

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | F1 Baja | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|---|
| Regresión logística | 0.600 | 0.768 | 0.560 | 0.694 | 0.560 | 0.525 | 0.865 | 0.368 |
| Regresión logística balanceada | 0.629 | 0.719 | 0.798 | 0.587 | 0.798 | 0.506 | 0.873 | 0.482 |
| Naive Bayes | 0.472 | 0.612 | 0.674 | 0.505 | 0.674 | 0.208 | 0.835 | 0.289 |
| KNN | 0.797 | 0.895 | 0.767 | 0.837 | 0.767 | 0.694 | 0.925 | 0.753 |
| SVM lineal (calibrado) | 0.516 | 0.753 | 0.479 | 0.668 | 0.479 | 0.358 | 0.874 | 0.246 |
| SVM RBF | 0.818 | 0.900 | 0.793 | 0.850 | 0.793 | 0.744 | 0.960 | 0.775 |

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| Ridge | 0.479 | 0.096 | 0.070 |
| Lasso | 0.479 | 0.096 | 0.069 |
| KNN | 0.796 | 0.060 | 0.036 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| SVR RBF | 0.806 | 0.059 | 0.037 |

El ajuste cambia poco a los modelos lineales y más a los no lineales:
- El SVR RBF sube de 0.734 a 0.806 en R². La búsqueda eligió un margen `epsilon` de 0.01 en lugar de 0.1, el valor por defecto. Con un índice que va de 0 a 0.94, un margen de 0.1 ignoraba errores grandes.
- KNN sube de 0.790 a 0.796 en R² y de 0.787 a 0.797 en F1, con 5 vecinos ponderados por distancia.
- Naive Bayes sube de 0.448 a 0.472 en F1 con más suavizado de la varianza.
- Los hiperparámetros elegidos son estables entre pliegues en KNN, Naive Bayes y SVR RBF, y en el SVM RBF el gamma fue `scale` en todos. En la regresión logística, Ridge y Lasso cambian entre pliegues sin que cambie el resultado, lo que indica una superficie plana: la regularización no es el cuello de botella.

![Benchmark mundial](fig_benchmark_modelos.png)

![ROC y matriz de confusión, base mundial](fig_benchmark_roc_confusion.png)

![Efecto del ajuste, base mundial](fig_benchmark_ajuste.png)

## R9. Resultados de la base de Colombia

Hiperparámetros ajustados con validación anidada (zona fuera y futuro de cada planta).

Regresión del FC diario:

| Modelo | R² dentro de planta (mediana) | Peor planta | Mejor planta | RMSE | MAE | MAE relativo |
|---|---|---|---|---|---|---|
| Referencia: nivel de la planta | -0.039 | -0.950 | 0.000 | 0.063 | 0.048 | 0.196 |
| Regresión lineal | 0.466 | -0.469 | 0.619 | 0.046 | 0.035 | 0.141 |
| Ridge | 0.461 | -0.473 | 0.613 | 0.046 | 0.035 | 0.141 |
| Lasso | 0.469 | -0.462 | 0.618 | 0.046 | 0.035 | 0.140 |
| KNN | 0.414 | -0.437 | 0.521 | 0.047 | 0.036 | 0.147 |
| SVR RBF | 0.407 | -0.292 | 0.528 | 0.047 | 0.035 | 0.144 |

Clasificación del día en bajo, medio o alto:

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|
| Referencia | 0.178 | 0.364 | 0.333 | 0.121 | 0.333 | 0.500 | 0.000 |
| Regresión logística | 0.572 | 0.576 | 0.574 | 0.570 | 0.574 | 0.765 | 0.575 |
| Naive Bayes | 0.535 | 0.544 | 0.540 | 0.538 | 0.540 | 0.737 | 0.508 |
| KNN | 0.564 | 0.571 | 0.566 | 0.567 | 0.566 | 0.753 | 0.561 |
| SVM lineal (calibrado) | 0.542 | 0.558 | 0.559 | 0.539 | 0.559 | 0.751 | 0.553 |
| SVM RBF | 0.572 | 0.576 | 0.570 | 0.580 | 0.570 | 0.759 | 0.568 |

Hiperparámetros elegidos: la búsqueda tomó siempre la opción más suave (KNN con 101 vecinos, SVR con C de 0.1, Ridge con alpha de 100). Con 6 zonas, lo que generaliza
es suavizar mucho. KNN y SVR quedaron en el borde de su grilla.

Contrastes (por defecto): con solo la radiación, el SVR sube de 0.358 a 0.488 en R². Con partición aleatoria, el SVR sube de 0.358 a 0.479 y el SVM RBF de 0.570 a 0.598 en F1.
Las demás variables de clima agregan ruido, y la partición aleatoria infla el resultado.

Prueba de fuentes de clima (cuaderno de clima): la radiación de `best_match` explica más que la de `era5` en 13 de las 16 plantas (mediana de R² de 0.49 frente a 0.41).
Parte del límite del resultado es la calidad de la radiación modelada.

![Benchmark de Colombia](fig_benchmark_colombia.png)

![ROC y matriz de confusión, Colombia](fig_benchmark_colombia_roc_confusion.png)

## R10. Tabla comparativa de todos los modelos

Todos con hiperparámetros ajustados por validación anidada. La mejor fila de cada tarea va primero.

Base mundial, clasificación de la clase de aptitud (bloques de 5°, con latitud y longitud):

| Modelo | F1 macro | Exactitud bal. | AUC | Kappa cuad. | F1 Baja |
|---|---|---|---|---|---|
| SVM RBF | 0.818 | 0.793 | 0.960 | 0.775 | 0.744 |
| KNN | 0.797 | 0.767 | 0.925 | 0.753 | 0.694 |
| Regresión logística balanceada | 0.629 | 0.798 | 0.873 | 0.482 | 0.506 |
| Regresión logística | 0.600 | 0.560 | 0.865 | 0.368 | 0.525 |
| SVM lineal (calibrado) | 0.516 | 0.479 | 0.874 | 0.246 | 0.358 |
| Naive Bayes | 0.472 | 0.674 | 0.835 | 0.289 | 0.208 |
| Referencia | 0.285 | 0.333 | 0.500 | 0.000 | 0.000 |

Base mundial, regresión del índice:

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| SVR RBF | 0.806 | 0.059 | 0.037 |
| KNN | 0.796 | 0.060 | 0.036 |
| Ridge | 0.479 | 0.096 | 0.070 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| Lasso | 0.479 | 0.096 | 0.069 |
| Referencia | -0.002 | 0.134 | 0.103 |

Colombia, regresión del factor de capacidad diario (zona fuera y futuro de cada planta):

| Modelo | R² dentro de planta (mediana) | RMSE | MAE relativo |
|---|---|---|---|
| Lasso | 0.469 | 0.046 | 0.140 |
| Regresión lineal | 0.466 | 0.046 | 0.141 |
| Ridge | 0.461 | 0.046 | 0.141 |
| KNN | 0.414 | 0.047 | 0.147 |
| SVR RBF | 0.407 | 0.047 | 0.144 |
| Referencia | -0.039 | 0.063 | 0.196 |

Colombia, clasificación del día (bajo, medio, alto):

| Modelo | F1 macro | Exactitud bal. | AUC | Kappa cuad. |
|---|---|---|---|---|
| Regresión logística | 0.572 | 0.574 | 0.765 | 0.575 |
| SVM RBF | 0.572 | 0.570 | 0.759 | 0.568 |
| KNN | 0.564 | 0.566 | 0.753 | 0.561 |
| SVM lineal (calibrado) | 0.542 | 0.559 | 0.751 | 0.553 |
| Naive Bayes | 0.535 | 0.540 | 0.737 | 0.508 |
| Referencia | 0.178 | 0.333 | 0.500 | 0.000 |

Lo que cambia entre bases: en la mundial ganan los modelos locales (SVM RBF, SVR RBF, KNN), porque el índice depende de la región. En Colombia ganan los lineales, porque la energía es casi proporcional a la radiación y hay pocas zonas para aprender algo más complejo.

![Comparativa de todos los modelos](fig_resumen_comparativa.png)

## R11. Interpretación: por qué unos modelos quedan bajos y cómo se podría mejorar

### Base mundial

Ridge, Lasso, la regresión logística y el SVM lineal quedan muy por debajo de KNN y del SVM con kernel (R² de 0.48 frente a 0.79; F1 de 0.60 frente a 0.81). La razón es la forma del problema, no la falta de ajuste:

- Un modelo lineal ajusta un único plano para todo el mundo. El índice tiene un nivel distinto en cada región y la relación con el terreno cambia de una región a otra. A eso se le llama no estacionariedad espacial, y un modelo global lineal no puede representarla (Brunsdon et al., 1996).
- KNN y el SVM con kernel ajustan vecindarios locales. Con latitud y longitud entre las variables, aprenden el nivel de cada región. Por eso son los que más caen al quitar latitud y longitud (KNN pasa de 0.79 a 0.48 en R²).
- Lasso no ayuda porque el problema no es que sobren variables. Con 15 variables y 58 mil filas la regularización casi no cambia nada. Con el ajuste, Lasso eligió valores de alpha muy pequeños (de 0.00001 a 0.001), es decir, casi sin penalización, y su R² quedó igual (0.479).
- Naive Bayes es el más bajo porque supone que las variables son independientes dentro de cada clase y con distribución normal. Aquí están correlacionadas (temperatura, humedad, radiación y latitud) y varias no son normales (Hastie et al., 2009).

Qué se podría hacer, según la literatura:
1. Ensambles de árboles: bosques aleatorios (Breiman, 2001) y XGBoost (Chen y Guestrin, 2016). Capturan interacciones y no linealidades sin que haya que especificarlas, y los pide la entrega siguiente. Lo esperable es un resultado parecido al de KNN y SVM con kernel dentro de las regiones conocidas.
2. Modelos espaciales: un bosque aleatorio con distancias o coordenadas como covariables (Hengl et al., 2018), o una regresión geográficamente ponderada, que estima coeficientes distintos en cada lugar y hace explícita la no estacionariedad (Brunsdon et al., 1996).
3. Stacking: combinar las predicciones de varios modelos (lineal, KNN y SVM) con un modelo de segundo nivel entrenado con predicciones fuera de pliegue (Wolpert, 1992).
4. Una advertencia: subir el puntaje dentro de las regiones conocidas no arregla la transferencia entre continentes ni hace que el índice mida aptitud. El área de aplicabilidad (Meyer y Pebesma, 2021) sirve para delimitar dónde una predicción es confiable. Quitar el efecto del lote de procesamiento exigiría recalcular el índice con datos homogéneos, no un mejor modelo.

### Base de Colombia

Aquí pasa lo contrario: los lineales son los mejores (R² de 0.47) y los flexibles quedan por debajo (0.41). La física explica por qué. La energía de una planta es casi proporcional a la radiación que recibe, así que un modelo lineal ya tiene la forma correcta. Con solo 6 zonas, los modelos flexibles aprenden detalles de esas zonas que no se repiten en otras. Por eso la búsqueda de hiperparámetros eligió siempre la opción más suave.

El techo de alrededor de 0.5 viene de los datos, no del modelo:
- La radiación es modelada, no medida en la planta. Cambiar de producto (`era5` a `best_match`) subió la mediana de 0.41 a 0.49. Urraca et al. (2018) evaluaron la radiación de ERA5 contra estaciones y productos satelitales. Su lectura completa está pendiente.
- Los promedios diarios esconden la variación de las nubes dentro del día.
- El nivel de cada planta depende de su tecnología y de recortes o mantenimiento que no medimos, y en algunas plantas cambia con el tiempo.

Qué se podría hacer, según la literatura:
1. Un modelo híbrido físico y estadístico: calcular la radiación en el plano de los paneles y la temperatura de celda con pvlib (Holmgren et al., 2018), con la tecnología real de cada planta, y usar el aprendizaje automático solo para corregir el residuo. Antonanzas et al. (2016) revisan los enfoques físicos, estadísticos e híbridos para pronóstico FV.
2. Mejor radiación: productos satelitales o mediciones en estaciones en lugar de reanálisis.
3. Modelos de efectos mixtos con un intercepto aleatorio por planta (Bates et al., 2015), o un bosque aleatorio de efectos mixtos (Hajjem et al., 2014). Manejan el nivel de cada planta y la dependencia entre días de la misma planta dentro del modelo, en lugar de centrar a mano.
4. Datos horarios y un índice de claridad, para separar el efecto de las nubes del de la geometría solar (Antonanzas et al., 2016).
5. Ensambles de árboles: con 6 zonas probablemente aportan poco, porque los modelos flexibles ya quedaron por debajo del lineal. Con más plantas y zonas podrían ayudar.
6. Completar la tecnología de cada planta, que es lo que más probablemente explica el 16 % de la varianza entre plantas.

## R12. Conclusiones

1. El índice de aptitud del dataset no sirve para predecir aptitud desde el clima. Su fórmula solo usa terreno, y en la práctica depende sobre todo de la región y del lote de procesamiento.
2. Los modelos base del curso predicen bien el índice dentro de las regiones conocidas. El SVM RBF llega a F1 macro de 0.82 y AUC de 0.96, y el SVR RBF a R² de 0.81. Pero lo logran en buena parte reconociendo la ubicación: sin latitud y longitud caen, y dejando una región fuera todos dan R² negativo.
3. Con la validación espacial por bloques los resultados salen más bajos que con partición aleatoria. Esa diferencia es la que justifica la validación espacial.
4. La base de Colombia, construida con generación real de XM y clima de Open-Meteo para 16 plantas, muestra que el clima del día explica cerca de la mitad del sube y baja diario de una planta (R² de 0.47 a 0.49 dentro de planta). Casi todo lo aporta la radiación, y cada kWh/m² suma unos 4.7 puntos de factor de capacidad, como predice la física.
5. En Colombia los modelos lineales son los mejores. El límite está en los datos: radiación modelada, promedios diarios, pocas zonas y la tecnología de cada planta sin medir.
6. El hyperparameter tuning con validación anidada mejora poco los modelos lineales y algo más los no lineales (el SVR RBF mundial de 0.73 a 0.81). En casi todos los casos el cuello de botella son los datos y no los hiperparámetros.

## R13. Limitaciones y pendientes

- La tecnología de cada planta colombiana (seguidor o fija, potencia DC frente a AC) no se ha investigado. Es lo que más probablemente explica el 16 % de la varianza entre plantas.
- El clima de Colombia es modelado, no medido. Falta comparar con radiación medida por estaciones del IDEAM.
- Las grillas de hiperparámetros son pequeñas. En Colombia, KNN y SVR quedaron en el borde de su grilla.
- No se usaron todavía árboles de decisión, bosques aleatorios ni XGBoost, ni balanceo con SMOTE o ADASYN, que pide la entrega siguiente.
- Los SVM con kernel de la base mundial se entrenaron con una submuestra de 8,000 filas.
- Las referencias se verificaron contra Crossref, DataCite o la página de la revista. Antes de citarlas en el informe final hay que leer los textos completos.

## R14. Referencias

- Antonanzas, J., Osorio, N., Escobar, R., Urraca, R. et al. (2016). Review of photovoltaic power forecasting. *Solar Energy, 136*, 78-111. https://doi.org/10.1016/j.solener.2016.06.069
- Bates, D., Mächler, M., Bolker, B. y Walker, S. (2015). Fitting linear mixed-effects models using lme4. *Journal of Statistical Software, 67*(1). https://doi.org/10.18637/jss.v067.i01
- Bergstra, J. y Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*, 281-305. https://jmlr.org/papers/v13/bergstra12a.html
- Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5-32. https://doi.org/10.1023/A:1010933404324
- Brunsdon, C., Fotheringham, A. S. y Charlton, M. E. (1996). Geographically weighted regression: a method for exploring spatial nonstationarity. *Geographical Analysis, 28*(4), 281-298. https://doi.org/10.1111/j.1538-4632.1996.tb00936.x
- Cawley, G. C. y Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079-2107. https://jmlr.org/papers/v11/cawley10a.html
- Chang, C.-C. y Lin, C.-J. (2011). LIBSVM: a library for support vector machines. *ACM Transactions on Intelligent Systems and Technology, 2*, 1-27. https://doi.org/10.1145/1961189.1961199
- Chen, T. y Guestrin, C. (2016). XGBoost: a scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785-794. https://doi.org/10.1145/2939672.2939785
- Cohen, J. (1968). Weighted kappa: nominal scale agreement with provision for scaled disagreement or partial credit. *Psychological Bulletin, 70*, 213-220. https://doi.org/10.1037/h0026256
- Cortes, C. y Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*, 273-297. https://doi.org/10.1007/BF00994018
- Cover, T. y Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory, 13*, 21-27. https://doi.org/10.1109/TIT.1967.1053964
- Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*, 861-874. https://doi.org/10.1016/j.patrec.2005.10.010
- Fisher, N. I. (1993). *Statistical analysis of circular data*. Cambridge University Press. https://doi.org/10.1017/CBO9780511564345
- Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *The Annals of Statistics*. https://doi.org/10.1214/aos/1013203451
- Global Energy Monitor. (2026). *Global Solar Power Tracker, February 2026 release*. https://globalenergymonitor.org/projects/global-solar-power-tracker
- Hajjem, A., Bellavance, F. y Larocque, D. (2014). Mixed-effects random forest for clustered data. *Journal of Statistical Computation and Simulation, 84*(6), 1313-1328. https://doi.org/10.1080/00949655.2012.741599
- Hand, D. J. y Till, R. J. (2001). A simple generalisation of the area under the ROC curve for multiple class classification problems. *Machine Learning, 45*, 171-186. https://doi.org/10.1023/A:1010920819831
- Hastie, T., Tibshirani, R. y Friedman, J. (2009). *The elements of statistical learning* (2.ª ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7
- He, H. y Garcia, E. A. (2009). Learning from imbalanced data. *IEEE Transactions on Knowledge and Data Engineering, 21*, 1263-1284. https://doi.org/10.1109/TKDE.2008.239
- Hengl, T., Nussbaum, M., Wright, M. N., Heuvelink, G. B. M. et al. (2018). Random forest as a generic framework for predictive modeling of spatial and spatio-temporal variables. *PeerJ, 6*, e5518. https://doi.org/10.7717/peerj.5518
- Hersbach, H. et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society, 146*(730), 1999-2049. https://doi.org/10.1002/qj.3803
- Hoerl, A. E. y Kennard, R. W. (1970). Ridge regression: biased estimation for nonorthogonal problems. *Technometrics, 12*, 55-67. https://doi.org/10.1080/00401706.1970.10488634
- Hollander, M., Wolfe, D. A. y Chicken, E. (2015). *Nonparametric statistical methods*. Wiley. https://doi.org/10.1002/9781119196037
- Holmgren, W. F., Hansen, C. W. y Mikofski, M. A. (2018). pvlib python: a python package for modeling solar energy systems. *Journal of Open Source Software, 3*(29), 884. https://doi.org/10.21105/joss.00884
- Kaufman, S., Rosset, S., Perlich, C. y Stitelman, O. (2012). Leakage in data mining: formulation, detection, and avoidance. *ACM Transactions on Knowledge Discovery from Data, 6*(4), 1-21. https://doi.org/10.1145/2382577.2382579
- Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J. y Garcia-Rodriguez, J. (2026). Global dataset of solar power plants: multidimensional integration and analysis. *Eng, 7*(7), 343. https://doi.org/10.3390/eng7070343
- Meyer, H. y Pebesma, E. (2021). Predicting into unknown space? Estimating the area of applicability of spatial prediction models. *Methods in Ecology and Evolution, 12*(9), 1620-1633. https://doi.org/10.1111/2041-210X.13650
- Niculescu-Mizil, A. y Caruana, R. (2005). Predicting good probabilities with supervised learning. *Proceedings of the 22nd International Conference on Machine Learning*, 625-632. https://doi.org/10.1145/1102351.1102430
- O'Brien, R. M. (2007). A caution regarding rules of thumb for variance inflation factors. *Quality and Quantity, 41*, 673-690. https://doi.org/10.1007/s11135-006-9018-6
- Pedregosa, F. et al. (2011). Scikit-learn: machine learning in Python. *Journal of Machine Learning Research, 12*, 2825-2830. https://jmlr.org/papers/v12/pedregosa11a.html
- Ploton, P. et al. (2020). Spatial validation reveals poor predictive performance of large-scale ecological mapping models. *Nature Communications, 11*, 4540. https://doi.org/10.1038/s41467-020-18321-y
- Roberts, D. R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913-929. https://doi.org/10.1111/ecog.02881
- Smola, A. J. y Schölkopf, B. (2004). A tutorial on support vector regression. *Statistics and Computing, 14*, 199-222. https://doi.org/10.1023/B:STCO.0000035301.49549.88
- Sokolova, M. y Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. *Information Processing and Management, 45*, 427-437. https://doi.org/10.1016/j.ipm.2009.03.002
- Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society B, 58*, 267-288. https://doi.org/10.1111/j.2517-6161.1996.tb02080.x
- Urraca, R. et al. (2018). Evaluation of global horizontal irradiance estimates from ERA5 and COSMO-REA6 reanalyses using ground and satellite-based data. *Solar Energy, 164*, 339-354. https://doi.org/10.1016/j.solener.2018.02.059
- Varma, S. y Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics, 7*, 91. https://doi.org/10.1186/1471-2105-7-91
- Wolpert, D. H. (1992). Stacked generalization. *Neural Networks, 5*(2), 241-259. https://doi.org/10.1016/S0893-6080(05)80023-1
- XM S.A. E.S.P. (2026). *API pública de datos del mercado de energía mayorista y portal SiMEM*. https://www.simem.co/
- Zippenfenig, P. (2023). *Open-Meteo.com Weather API* [software]. Zenodo. https://doi.org/10.5281/zenodo.7970649

## R15. Terminología

| Término | Qué es | Para qué se usó en este proyecto |
|---|---|---|
| Índice de aptitud solar (IAS, `solar_aptitude`) | Índice de 0 a 1 que el paper calcula con pendiente, orientación, sombreado y curvatura | Variable objetivo de la base mundial, en regresión y en sus tres clases |
| Factor de capacidad (FC) | Energía generada dividida entre la máxima posible a potencia nominal en el mismo tiempo | Variable objetivo de la base de Colombia |
| Capacidad efectiva neta | Potencia máxima que la planta puede entregar a la red, según XM | Denominador del factor de capacidad |
| Potencia DC y AC | Potencia pico de los paneles (DC) y la que sale del inversor (AC) | Su relación es parte de la tecnología pendiente de cada planta |
| Seguidor solar | Estructura que gira los paneles para seguir al sol | Tecnología pendiente; cambia el nivel de FC de cada planta |
| Radiación global horizontal (GHI) | Energía solar que llega a una superficie horizontal, en kWh/m² por día | Principal variable de clima en la base de Colombia |
| Nubosidad | Fracción del cielo cubierta por nubes | Variable de clima; se probó si aporta además de la radiación |
| Reanálisis (ERA5) | Reconstrucción del clima pasado que combina modelo y observaciones | Fuente del clima; se comparó con `best_match` |
| `best_match` | Opción por defecto de Open-Meteo, que combina IFS HRES, ERA5 y ERA5-Land | Fuente de clima que mejor explicó la generación |
| NOCT | Temperatura de operación nominal de una celda, usada para estimarla desde la temperatura del aire | Línea base física con pérdida por temperatura |
| Dato centinela | Valor que en realidad significa "sin dato" (un 0 o un -1) | Se detectaron y se eliminaron o recodificaron en el EDA |
| Fuga de información (data leakage) | Que el modelo use, al entrenar, información que no tendría en la práctica, como los datos de prueba | Se evitó con la partición primero y todo el preprocesamiento dentro del `Pipeline` |
| `Pipeline` | Cadena de pasos de scikit-learn que se ajustan juntos solo con el entrenamiento | Imputación, escalado, calibración y modelo en un solo objeto |
| Imputación por mediana | Rellenar un vacío con la mediana del entrenamiento | 63 vacíos de distancia a carretera en la base mundial |
| Estandarización | Restar la media y dividir entre la desviación estándar | Necesaria para KNN, SVM y modelos regularizados |
| Validación cruzada | Repetir entrenamiento y prueba en varias particiones | Base de toda la evaluación |
| Validación espacial por bloques | Partir por zonas geográficas enteras en vez de filas al azar | Evitar que plantas vecinas queden en entrenamiento y prueba a la vez |
| Partición temporal | Entrenar con el pasado y probar con el futuro | Base de Colombia: el nivel de cada planta sale de su pasado |
| Validación anidada | Bucle interno que elige hiperparámetros y bucle externo que mide | hyperparameter tuning sin inflar el resultado |
| Hiperparámetro | Parámetro que no se aprende de los datos y se fija antes (C, alpha, k) | Se ajustaron con búsqueda en grilla |
| Búsqueda en grilla | Probar todas las combinaciones de una lista de valores | hyperparameter tuning |
| Regularización (L1, L2) | Penalización que limita los coeficientes del modelo | Ridge (L2), Lasso (L1) y el parámetro C de logística y SVM |
| Kernel RBF | Función que permite al SVM separar datos con fronteras no lineales | SVM y SVR con kernel |
| Calibración de probabilidades | Ajustar las salidas del modelo para que se comporten como probabilidades | Probabilidades del SVM lineal, necesarias para el AUC |
| Clase desbalanceada | Clase con muchas menos observaciones que las demás | La clase Baja (3 %) en la base mundial |
| `class_weight="balanced"` | Dar más peso en el entrenamiento a las clases poco frecuentes | Regresión logística balanceada |
| F1 macro | Promedio simple del F1 de cada clase | Métrica principal de clasificación |
| AUC ROC | Probabilidad de que el modelo ordene bien un ejemplo positivo frente a uno negativo | Medir la calidad de las probabilidades sin depender de un umbral |
| Kappa ponderado cuadrático | Acuerdo entre predicción y realidad que castiga más los errores entre clases lejanas | Clases con orden (Baja, Media, Alta; bajo, medio, alto) |
| R² | Proporción de la varianza explicada por el modelo | Métrica principal de regresión |
| RMSE y MAE | Raíz del error cuadrático medio y error absoluto medio | Magnitud del error en unidades de la variable |
| R² dentro de planta | R² calculado con la variable centrada en la media de cada planta | Medir cuánto explica el clima el sube y baja diario, sin el nivel de cada planta |
| Leave-region-out | Entrenar sin una región entera y probar en ella | Mostrar que el modelo del índice no se transfiere entre continentes |
| Bootstrap por planta | Remuestrear plantas, no filas, para obtener intervalos de confianza | Intervalos del R² en el cuaderno de clima |
| Prueba de Wilcoxon | Prueba no paramétrica para diferencias pareadas | Comparar modelos planta por planta en el cuaderno de clima |